<h3><center> GPT Recreation Using The Multi Head Attention Model </h3>

In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F 
import matplotlib.pyplot as plt

In [20]:
txt = open('shakespere.txt', 'r', encoding='utf-8')
text = txt.read()
len(text)

1115393

In [21]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size, ''.join(chars))

65 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [22]:
stoi = {i:s for s, i in enumerate(chars)}
itos = {s:i for i, s in stoi.items()}

encode = lambda s : [stoi[c] for c in s]
decode = lambda x : ''.join([itos[c] for c in x])

In [23]:
print(encode('hello'))
print(decode(encode('hello')))

[46, 43, 50, 50, 53]
hello


In [24]:
# Embedding character encodes into tensor

data = torch.tensor(encode(text))
data[:100]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

In [25]:
# Train-test split

k = int(0.8*len(data))
train_dat = data[:k]
test_dat = data[k:]
print(len(train_dat), len(test_dat), len(data))

892314 223079 1115393


In [26]:
# Initialisation

batch_size = 4      # Number of independent squences for batch processing   --> aka batch(B)
block_size = 8      # The context length of the model                       --> aka time(T)
torch.manual_seed(982734)

In [27]:
# Batch dimension

def get_batch(split):
    d = train_dat if split == 'train' else test_dat
    ix = torch.randint(len(d) - block_size, (batch_size, ))
    # print(ix.shape)
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print(xb)

tensor([[ 1, 58, 53,  1, 31, 39, 52, 42],
        [ 5, 57,  1, 57, 53, 52,  6,  0],
        [ 1, 58, 53,  1, 58, 46, 63,  1],
        [42, 47, 42,  1, 58, 53,  1, 15]])


In [28]:
# Just to see what the model is reading in a batch

oi = xb.tolist()
xbdecoded = [decode(o) for o in oi]
xbdecoded

[' to Sand', "'s son,\n", ' to thy ', 'did to C']

In [29]:
xb , yb

(tensor([[ 1, 58, 53,  1, 31, 39, 52, 42],
         [ 5, 57,  1, 57, 53, 52,  6,  0],
         [ 1, 58, 53,  1, 58, 46, 63,  1],
         [42, 47, 42,  1, 58, 53,  1, 15]]),
 tensor([[58, 53,  1, 31, 39, 52, 42, 39],
         [57,  1, 57, 53, 52,  6,  0, 13],
         [58, 53,  1, 58, 46, 63,  1, 40],
         [47, 42,  1, 58, 53,  1, 15, 46]]))

In [30]:
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"{context.tolist()} ------> {target}")

[1] ------> 58
[1, 58] ------> 53
[1, 58, 53] ------> 1
[1, 58, 53, 1] ------> 31
[1, 58, 53, 1, 31] ------> 39
[1, 58, 53, 1, 31, 39] ------> 52
[1, 58, 53, 1, 31, 39, 52] ------> 42
[1, 58, 53, 1, 31, 39, 52, 42] ------> 39
[5] ------> 57
[5, 57] ------> 1
[5, 57, 1] ------> 57
[5, 57, 1, 57] ------> 53
[5, 57, 1, 57, 53] ------> 52
[5, 57, 1, 57, 53, 52] ------> 6
[5, 57, 1, 57, 53, 52, 6] ------> 0
[5, 57, 1, 57, 53, 52, 6, 0] ------> 13
[1] ------> 58
[1, 58] ------> 53
[1, 58, 53] ------> 1
[1, 58, 53, 1] ------> 58
[1, 58, 53, 1, 58] ------> 46
[1, 58, 53, 1, 58, 46] ------> 63
[1, 58, 53, 1, 58, 46, 63] ------> 1
[1, 58, 53, 1, 58, 46, 63, 1] ------> 40
[42] ------> 47
[42, 47] ------> 42
[42, 47, 42] ------> 1
[42, 47, 42, 1] ------> 58
[42, 47, 42, 1, 58] ------> 53
[42, 47, 42, 1, 58, 53] ------> 1
[42, 47, 42, 1, 58, 53, 1] ------> 15
[42, 47, 42, 1, 58, 53, 1, 15] ------> 46


In [42]:
# Feeding the characters in biagram language model that I created in the previous projects

class BiagramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # Reading the logits off of the next token from the lookup table and then traning on that model
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx = B
        # targets = T
        # logits = combination of BTC
        logits = self.token_embedding_table(idx)

        # Reshaping logits
        if targets == None:
            loss = None                     # For generating
        else:
            B, T, C = logits.shape          # For training
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is the current context in a batch
        # makes B X T+1, B X T+2, B X T+3 and so on..
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]                           # (B, C)
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)   
            idx = torch.cat((idx, idx_next), dim=1)             # (B, T+1)
        
        return idx

m = BiagramLanguageModel(vocab_size=vocab_size)
logits, loss = m(xb , yb)
print(logits.shape, loss.shape)

torch.Size([32, 65]) torch.Size([])


In [54]:
# Generating from a random model
idx1 = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx1, max_new_tokens=100)[0].tolist()))


fdiJ lci;ytSOjhUUPwPqNU3dW:PD's;iXRjPW:kAtS;&xQ$
3tSGvxG
nPjQaiF$tS;BSz.c3YtgPTZklZZTgDcOBWlM'gOcn -


<h3><center> Traning the model</h3>

In [ ]:
# Creating an optimiser

